In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Kafka-Streaming") \
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages",",".join([ "org.apache.hadoop:hadoop-aws:3.3.4", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"])) \
    .config("spark.hadoop.fs.s3a.endpoint","http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key","admin") \
    .config("spark.hadoop.fs.s3a.secret.key","password123") \
    .config("spark.hadoop.fs.s3a.path.style.access","true") \
    .config("spark.hadoop.fs.s3a.impl","org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"]).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1f8b843f-63d9-4af6-8c8a-71d8fffcd56d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 

In [2]:
# Read Kafka Stream - spark.readStream(Keep listening forever)
stream_df = spark.readStream \
    .format("kafka").option("kafka.bootstrap.servers","kafka:9092") \
    .option("subscribe","user-events") \
    .load()

In [3]:
# Parse Messages
from pyspark.sql.functions import col

parsed_df = stream_df.select(
    col("value").cast("string").alias("json_data")
)

In [6]:
# Start Console Streaming
query = parsed_df.writeStream.format("delta") \
    .outputMode("append").option("checkpointLocation","s3a://bronze/checkpoints/user-events") \
    .start("s3a://bronze/user-events-delta")

query.awaitTermination()

26/05/22 13:57:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/22 13:57:23 WARN StreamingQueryManager: Stopping existing streaming query [id=172704bb-e346-4e3f-b855-f318de350240, runId=5cae2657-4d4d-48b3-a2f7-4b31ae034f59], as a new run is being started.
26/05/22 13:57:23 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/05/22 13:58:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/us

KeyboardInterrupt: 

In [7]:
df = spark.read.format("delta").load(
    "s3a://bronze/user-events-delta"
)

df.show(truncate=False)

+--------------------------------------------------------+
|value                                                   |
+--------------------------------------------------------+
|{"user_id": 5, "action": "login", "amount": 177}        |
|{"user_id": 96, "action": "purchase", "amount": 489}    |
|{"user_id": 5, "action": "add_to_cart", "amount": 68}   |
|{"user_id": 59, "action": "login", "amount": 75}        |
|{"user_id": 12, "action": "add_to_cart", "amount": 67}  |
|{"user_id": 65, "action": "purchase", "amount": 29}     |
|{"user_id": 90, "action": "logout", "amount": 217}      |
|{"user_id": 63, "action": "purchase", "amount": 359}    |
|{"user_id": 80, "action": "logout", "amount": 272}      |
|{"user_id": 17, "action": "login", "amount": 98}        |
|{"user_id": 55, "action": "login", "amount": 136}       |
|{"user_id": 5, "action": "add_to_cart", "amount": 281}  |
|{"user_id": 20, "action": "logout", "amount": 498}      |
|{"user_id": 100, "action": "add_to_cart", "amount": 108

In [9]:
df.rdd.getNumPartitions()

3